In [1]:
# import osmnx as ox
# import networkx as nx
# import pandas as pd
# import numpy as np
# import os
# import time

# # ==============================
# # CONFIG
# # ==============================
# OUTPUT_CSV = "synthetic_full_vasai_TEMPORAL.csv"
# BATCH_SIZE = 2000
# DAYS = 7
# DIST = 6000

# # ==============================
# # 1. DOWNLOAD GRAPH
# # ==============================
# print("\n[STEP 1] Downloading Mumbai...\n")
# lats = np.arange(19.00, 19.45, 0.05)
# lons = np.arange(72.80, 73.10, 0.05)
# graphs = []

# for lat in lats:
#     for lon in lons:
#         for _ in range(3):
#             try:
#                 g = ox.graph_from_point((lat, lon), dist=DIST, network_type="drive")
#                 graphs.append(g)
#                 break
#             except:
#                 time.sleep(1)

# G = nx.compose_all(graphs)

# # ==============================
# # 2. EXTRACT DATA
# # ==============================
# nodes, edges = ox.graph_to_gdfs(G)
# edges = edges.reset_index()
# nodes = nodes[["x", "y"]]

# edges["u_lon"] = edges["u"].map(nodes["x"])
# edges["u_lat"] = edges["u"].map(nodes["y"])
# edges["v_lon"] = edges["v"].map(nodes["x"])
# edges["v_lat"] = edges["v"].map(nodes["y"])

# # ==============================
# # CLEAN
# # ==============================
# def normalize(x):
#     if isinstance(x, (list, np.ndarray)):
#         return x[0] if len(x) > 0 else None
#     return x

# def parse_speed(x):
#     x = normalize(x)
#     # ✅ THE FIX: guard against None AND nan before calling float()
#     # float(str(np.nan)) == float("nan") == nan — no exception is raised!
#     if x is None or (isinstance(x, float) and np.isnan(x)):
#         return 40.0
#     try:
#         return float(str(x).replace(" km/h", "").strip())
#     except:
#         return 40.0

# edges["maxspeed"] = edges["maxspeed"].apply(parse_speed)
# edges["free_flow_speed"] = edges["maxspeed"]

# # ✅ Sanity check — catch any remaining NaN before batch loop
# nan_count = edges["free_flow_speed"].isna().sum()
# print(f"[CHECK] free_flow_speed NaN count after parsing: {nan_count}")
# edges["free_flow_speed"] = edges["free_flow_speed"].fillna(40.0)

# road_weight = {
#     "motorway": 1.0,
#     "trunk": 0.9,
#     "primary": 0.8,
#     "secondary": 0.7,
#     "tertiary": 0.6,
#     "residential": 0.5
# }
# edges["road_weight"] = edges["highway"].apply(
#     lambda x: road_weight.get(normalize(x), 0.5)
# )

# # ==============================
# # TIME
# # ==============================
# timestamps = pd.date_range(start="2024-01-01", periods=24 * DAYS, freq="1h")
# hours = timestamps.hour.values
# days = timestamps.weekday.values
# T = len(timestamps)

# time_factor = (
#     0.75 + 0.25 * np.sin(2 * np.pi * (hours - 6) / 24)
# ) * (
#     0.85 + 0.15 * np.cos(2 * np.pi * days / 7)
# )
# time_factor = time_factor[None, :]  # shape (1, T)

# # ==============================
# # GENERATE DATA
# # ==============================
# if os.path.exists(OUTPUT_CSV):
#     os.remove(OUTPUT_CSV)

# edge_features = edges[[
#     "u", "v", "length", "free_flow_speed", "road_weight",
#     "u_lat", "u_lon", "v_lat", "v_lon"
# ]].to_numpy()

# for i in range(0, len(edge_features), BATCH_SIZE):
#     print(f"\n🚀 Batch {i} → {min(i+BATCH_SIZE, len(edge_features))}")

#     batch = edge_features[i:i+BATCH_SIZE]

#     u      = batch[:, 0]
#     v      = batch[:, 1]
#     length = batch[:, 2][:, None]        # shape (N, 1)
#     free_speed = batch[:, 3][:, None]    # shape (N, 1)
#     road_w     = batch[:, 4][:, None]    # shape (N, 1)

#     u_lat  = batch[:, 5]
#     u_lon  = batch[:, 6]
#     v_lat  = batch[:, 7]
#     v_lon  = batch[:, 8]

#     N = len(batch)

#     free_speed_vec = free_speed.flatten()   # shape (N,)
#     road_w_vec     = road_w.flatten()       # shape (N,)

#     # --------------------------
#     # TEMPORAL RAIN & INCIDENT
#     # --------------------------
#     rain     = np.zeros((N, T))
#     incident = np.zeros((N, T))

#     rain[:, 0]     = np.random.choice([0, 1], size=N, p=[0.8, 0.2])
#     incident[:, 0] = np.random.choice([0, 1], size=N, p=[0.95, 0.05])

#     for t in range(1, T):
#         rain[:, t] = np.where(
#             np.random.rand(N) < 0.9,
#             rain[:, t-1],
#             np.random.choice([0, 1], size=N, p=[0.8, 0.2])
#         )
#         incident[:, t] = np.where(
#             np.random.rand(N) < 0.85,
#             incident[:, t-1],
#             np.random.choice([0, 1], size=N, p=[0.95, 0.05])
#         )

#     # --------------------------
#     # CORRELATED NOISE
#     # --------------------------
#     noise = np.zeros((N, T))
#     noise[:, 0] = np.random.normal(1.0, 0.05, size=N)

#     for t in range(1, T):
#         noise[:, t] = 0.8 * noise[:, t-1] + 0.2 * np.random.normal(1.0, 0.05, size=N)

#     # --------------------------
#     # TEMPORAL SPEED
#     # --------------------------
#     speed = np.zeros((N, T))
#     speed[:, 0] = free_speed_vec * 0.8

#     for t in range(1, T):
#         weather    = np.where(rain[:, t] == 1, 0.75, 1.0)
#         incident_f = np.where(incident[:, t] == 1, 0.6, 1.0)

#         base = (
#             free_speed_vec
#             * time_factor[0, t]
#             * road_w_vec
#             * weather
#             * incident_f
#         )

#         delta = (base * noise[:, t] - speed[:, t-1])
#         speed[:, t] = speed[:, t-1] + 0.3 * delta

#     speed = np.clip(speed, 5, free_speed_vec[:, None])

#     # --------------------------
#     # DERIVED FEATURES
#     # --------------------------
#     congestion   = 1 - (speed / free_speed_vec[:, None])
#     travel_time  = length / (speed * 1000 / 3600)

#     # ✅ Final NaN check per batch — fail fast rather than silently write NaN
#     assert not np.isnan(speed).any(),       "NaN in speed!"
#     assert not np.isnan(congestion).any(),  "NaN in congestion!"
#     assert not np.isnan(travel_time).any(), "NaN in travel_time!"

#     df = pd.DataFrame({
#         "u":               np.repeat(u, T),
#         "v":               np.repeat(v, T),
#         "u_lat":           np.repeat(u_lat, T),
#         "u_lon":           np.repeat(u_lon, T),
#         "v_lat":           np.repeat(v_lat, T),
#         "v_lon":           np.repeat(v_lon, T),
#         "length_m":        np.repeat(length.flatten(), T),
#         "timestamp":       np.tile(timestamps, N),
#         "hour":            np.tile(hours, N),
#         "day_of_week":     np.tile(days, N),
#         "current_speed":   speed.flatten(),
#         "free_flow_speed": np.repeat(free_speed_vec, T),
#         "congestion":      congestion.flatten(),
#         "travel_time_sec": travel_time.flatten(),
#         "rain":            rain.flatten(),
#         "incident":        incident.flatten()
#     })

#     df.to_csv(OUTPUT_CSV, mode='a', header=(i == 0), index=False, float_format="%.3f")
#     print("   💾 Saved")

# print("\n🎉 DONE (TEMPORAL DATA GENERATED)\n")
# ============================================================
# generate_synthetic_data.py
# ============================================================
import osmnx as ox
import networkx as nx
import pandas as pd
import numpy as np
import os
import time

OUTPUT_CSV = "synthetic_TEMPORAL.csv"
BATCH_SIZE = 2000
DAYS       = 7
DIST       = 6000

# ── 1. Download graph ────────────────────────────────────────
print("\n[STEP 1] Downloading Mumbai road network...\n")
lats   = np.arange(19.00, 19.45, 0.05)
lons   = np.arange(72.80, 73.10, 0.05)
graphs = []

for lat in lats:
    for lon in lons:
        for attempt in range(3):
            try:
                g = ox.graph_from_point((lat, lon), dist=DIST, network_type="drive")
                graphs.append(g)
                break
            except Exception:
                time.sleep(1)

G = nx.compose_all(graphs)
print(f"[INFO] Graph: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges")

# ── 2. Extract edges ─────────────────────────────────────────
nodes, edges = ox.graph_to_gdfs(G)
edges = edges.reset_index()
nodes = nodes[["x", "y"]]

edges["u_lon"] = edges["u"].map(nodes["x"])
edges["u_lat"] = edges["u"].map(nodes["y"])
edges["v_lon"] = edges["v"].map(nodes["x"])
edges["v_lat"] = edges["v"].map(nodes["y"])

# ── 3. Parse speed ───────────────────────────────────────────
def normalize(x):
    if isinstance(x, (list, np.ndarray)):
        return x[0] if len(x) > 0 else None
    return x

def parse_speed(x):
    x = normalize(x)
    if x is None or (isinstance(x, float) and np.isnan(x)):
        return 40.0
    try:
        return float(str(x).replace(" km/h", "").strip())
    except Exception:
        return 40.0

edges["maxspeed"]       = edges["maxspeed"].apply(parse_speed)
edges["free_flow_speed"] = edges["maxspeed"]
edges["free_flow_speed"] = edges["free_flow_speed"].fillna(40.0)

nan_count = edges["free_flow_speed"].isna().sum()
assert nan_count == 0, f"Still {nan_count} NaN in free_flow_speed!"
print(f"[CHECK] free_flow_speed NaN: {nan_count}")

road_weight_map = {
    "motorway": 1.0, "trunk": 0.9, "primary": 0.8,
    "secondary": 0.7, "tertiary": 0.6, "residential": 0.5
}
edges["road_weight"] = edges["highway"].apply(
    lambda x: road_weight_map.get(normalize(x), 0.5)
)

# ── 4. Timestamps ────────────────────────────────────────────
timestamps = pd.date_range(start="2024-01-01", periods=24 * DAYS, freq="1h")
hours      = timestamps.hour.values        # (T,)
days       = timestamps.weekday.values     # (T,)
T          = len(timestamps)

# ── 5. Generate in batches ───────────────────────────────────
if os.path.exists(OUTPUT_CSV):
    os.remove(OUTPUT_CSV)

edge_features = edges[[
    "u", "v", "length", "free_flow_speed", "road_weight",
    "u_lat", "u_lon", "v_lat", "v_lon"
]].to_numpy()

total_edges = len(edge_features)
print(f"\n[INFO] Generating data for {total_edges} edges × {T} timesteps...\n")

for i in range(0, total_edges, BATCH_SIZE):
    batch = edge_features[i : i + BATCH_SIZE]
    N = len(batch)
    print(f"  Batch {i}–{i+N} / {total_edges}")

    u          = batch[:, 0]
    v          = batch[:, 1]
    length     = batch[:, 2]          # (N,)
    free_speed = batch[:, 3]          # (N,)
    road_w     = batch[:, 4]          # (N,)
    u_lat      = batch[:, 5]
    u_lon      = batch[:, 6]
    v_lat      = batch[:, 7]
    v_lon      = batch[:, 8]

    # ── Per-road time factor ──────────────────────────────────
    # Each road gets its own phase shift + amplitude so the embedding
    # table has genuinely road-specific patterns to learn.
    # Without this, speed_ratio normalisation makes all roads look
    # statistically identical and the embedding learns nothing.

    phase_shift = np.random.uniform(-3, 3, size=N)      # hours offset
    amplitude   = np.random.uniform(0.10, 0.35, size=N) # peak variation depth
    road_bias   = np.random.uniform(0.65, 1.00, size=N) # per-road speed baseline

    # (N, T) — each road has its own sinusoidal daily curve
    hour_grid = hours[None, :]                          # (1, T)
    day_grid  = days[None, :]                           # (1, T)

    time_factor = (
        road_bias[:, None]
        * (0.75 + amplitude[:, None]
           * np.sin(2 * np.pi * (hour_grid - 6 + phase_shift[:, None]) / 24))
        * (0.85 + 0.15 * np.cos(2 * np.pi * day_grid / 7))
    )
    # Clamp so speed never mathematically exceeds free_flow
    time_factor = np.clip(time_factor, 0.1, 1.0)       # (N, T)

    # ── Rain & incident (AR(1) correlated) ───────────────────
    rain     = np.zeros((N, T))
    incident = np.zeros((N, T))
    rain[:, 0]     = np.random.choice([0, 1], size=N, p=[0.8, 0.2])
    incident[:, 0] = np.random.choice([0, 1], size=N, p=[0.95, 0.05])

    for t in range(1, T):
        rain[:, t]     = np.where(np.random.rand(N) < 0.9,
                                  rain[:, t-1],
                                  np.random.choice([0, 1], N, p=[0.8, 0.2]))
        incident[:, t] = np.where(np.random.rand(N) < 0.85,
                                  incident[:, t-1],
                                  np.random.choice([0, 1], N, p=[0.95, 0.05]))

    # ── AR(1) correlated noise ────────────────────────────────
    noise = np.zeros((N, T))
    noise[:, 0] = np.random.normal(1.0, 0.04, size=N)
    for t in range(1, T):
        noise[:, t] = 0.75 * noise[:, t-1] + 0.25 * np.random.normal(1.0, 0.04, size=N)

    # ── Speed simulation ──────────────────────────────────────
    speed = np.zeros((N, T))
    speed[:, 0] = free_speed * road_bias * 0.8

    for t in range(1, T):
        weather_f  = np.where(rain[:, t] == 1, 0.75, 1.0)
        incident_f = np.where(incident[:, t] == 1, 0.60, 1.0)

        target = (
            free_speed
            * time_factor[:, t]
            * road_w
            * weather_f
            * incident_f
            * noise[:, t]
        )

        # Smooth transition toward target (inertia)
        speed[:, t] = speed[:, t-1] + 0.3 * (target - speed[:, t-1])

    speed = np.clip(speed, 5.0, free_speed[:, None])

    # ── Derived features ──────────────────────────────────────
    congestion  = 1.0 - (speed / free_speed[:, None])
    congestion  = np.clip(congestion, 0.0, 1.0)
    travel_time = (length[:, None]) / (speed * 1000 / 3600 + 1e-6)

    assert not np.isnan(speed).any(),       "NaN in speed!"
    assert not np.isnan(congestion).any(),  "NaN in congestion!"
    assert not np.isnan(travel_time).any(), "NaN in travel_time!"

    # ── Assemble DataFrame ────────────────────────────────────
    df = pd.DataFrame({
        "u":               np.repeat(u, T),
        "v":               np.repeat(v, T),
        "u_lat":           np.repeat(u_lat, T),
        "u_lon":           np.repeat(u_lon, T),
        "v_lat":           np.repeat(v_lat, T),
        "v_lon":           np.repeat(v_lon, T),
        "length_m":        np.repeat(length, T),
        "timestamp":       np.tile(timestamps, N),
        "hour":            np.tile(hours, N),
        "day_of_week":     np.tile(days, N),
        "current_speed":   speed.flatten(),
        "free_flow_speed": np.repeat(free_speed, T),
        "congestion":      congestion.flatten(),
        "travel_time_sec": travel_time.flatten(),
        "rain":            rain.flatten(),
        "incident":        incident.flatten(),
    })

    df.to_csv(OUTPUT_CSV, mode="a", header=(i == 0), index=False, float_format="%.4f")
    print(f"    Saved — speed_ratio range: "
          f"[{(speed/free_speed[:,None]).min():.3f}, "
          f"{(speed/free_speed[:,None]).max():.3f}]")

print("\n[DONE] Data generation complete.\n")


[STEP 1] Downloading Mumbai road network...

[INFO] Graph: 62059 nodes, 143909 edges
[CHECK] free_flow_speed NaN: 0

[INFO] Generating data for 143909 edges × 168 timesteps...

  Batch 0–2000 / 143909
    Saved — speed_ratio range: [0.125, 0.800]
  Batch 2000–4000 / 143909
    Saved — speed_ratio range: [0.125, 0.976]
  Batch 4000–6000 / 143909
    Saved — speed_ratio range: [0.125, 0.965]
  Batch 6000–8000 / 143909
    Saved — speed_ratio range: [0.117, 0.941]
  Batch 8000–10000 / 143909
    Saved — speed_ratio range: [0.100, 0.903]
  Batch 10000–12000 / 143909
    Saved — speed_ratio range: [0.125, 0.905]
  Batch 12000–14000 / 143909
    Saved — speed_ratio range: [0.100, 0.897]
  Batch 14000–16000 / 143909
    Saved — speed_ratio range: [0.125, 0.882]
  Batch 16000–18000 / 143909
    Saved — speed_ratio range: [0.125, 0.863]
  Batch 18000–20000 / 143909
    Saved — speed_ratio range: [0.112, 0.931]
  Batch 20000–22000 / 143909
    Saved — speed_ratio range: [0.083, 0.869]
  Batch 2

In [2]:
import pandas as pd
data=pd.read_csv('D:/Urban-Traffic-and-Parking-Analysis-using-LSTM-Autoencoders-and-Reinforcement-Learning/Synthetic Data Generation/synthetic_TEMPORAL.csv')

In [3]:
len(data)

24176712

In [ ]:
data.head()

In [ ]:
data.head(-10)